In [ ]:
import os
import sys
parent_dir = os.path.abspath('..')
sys.path.append(parent_dir) #set root to path so src visible

import pandas as pd
from src.data_processing.data_processing import DataProcessor

!pip install openpyxl

train_processor = DataProcessor(mode="training")
df_train = train_processor.run() 

: 

In [ ]:
df_revenu = pd.read_excel(io="data_external/revenu_median_communes.xlsx")

In [ ]:
# STEP 1 : Nettoyer les revenus invalides
invalid = ["N/A - secret statistique", "N/A - résultat non disponible"]
df_revenu["Médiane du niveau de vie 2021"] = pd.to_numeric(
    df_revenu["Médiane du niveau de vie 2021"].replace(invalid, pd.NA), errors="coerce"
)

# STEP 2 : Exploser les codes postaux multiples (ex: "06130/06520" → 2 lignes)
df_revenu["Code postal"] = df_revenu["Code postal"].astype(str).str.split("/")
df_revenu = df_revenu.explode("Code postal")
df_revenu["Code postal"] = df_revenu["Code postal"].str.strip().str.zfill(5)

# STEP 3 : Remplacer les revenus manquants par la médiane des communes voisines (même préfixe CP)
df_revenu["_cp_prefix"] = df_revenu["Code postal"].str[:3]
prefix_median = (df_revenu.groupby("_cp_prefix")["Médiane du niveau de vie 2021"]
                          .median()
                          .rename("_revenu_voisin"))
df_revenu = df_revenu.merge(prefix_median, on="_cp_prefix", how="left")
mask = df_revenu["Médiane du niveau de vie 2021"].isna()
df_revenu.loc[mask, "Médiane du niveau de vie 2021"] = df_revenu.loc[mask, "_revenu_voisin"]
df_revenu = df_revenu.drop(columns=["_cp_prefix", "_revenu_voisin"])

# STEP 4 : Merger sur df_train
df_train["client_code_postal"] = df_train["client_code_postal"].astype(str).str.zfill(5)

new_column_name= "client_revenu_median_commune_2021"
df_train = (df_train
    .merge(df_revenu[["Code postal", "Médiane du niveau de vie 2021"]],
           left_on="client_code_postal", right_on="Code postal", how="left")
    .rename(columns={"Médiane du niveau de vie 2021": new_column_name})
    .drop(columns=["Code postal"])
)


In [ ]:
df_train["client_revenu_median_commune_2021"].isna().sum()/df_train.shape[0]*100
#2.52% revenus manquants --> OK données manquantes restante on remplace par le revenu médian en France

In [ ]:
revenu_median_fr= 23160
df_train["client_revenu_median_commune_2021"].fillna(revenu_median_fr)